# Stage 3 — VAE-A & VAE-B Combined Training
## Explainable Hybrid Quantum-Classical NIDS  |  v2 Dataset
Both models trained in a single notebook. Outputs saved to separate folders:
- `vae_a_output/` — WITH near-zero variance (167 features, VAE-A)
- `vae_b_output/` — WITHOUT near-zero variance (140 features, VAE-B)

**New in v2**: Class weights from Stage 2 applied to aux classifier loss.  
`kagglehub` used for dataset download.

## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

pkgs = ['kagglehub', 'torch', 'pandas==2.2.2', 'pyarrow==15.0.2',
        'numpy', 'matplotlib', 'scikit-learn']
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   capture_output=True)

print("All packages ready.")

All packages ready.


## Cell 2 — Download Dataset via kagglehub

In [2]:
import os, shutil

notebook_dir = os.getcwd()

# ── Place kaggle.json next to this notebook ──────────────────────────────────
kaggle_json_src = os.path.join(notebook_dir, 'kaggle.json')
if not os.path.exists(kaggle_json_src):
    raise FileNotFoundError(
        "kaggle.json not found in notebook directory!\n"
        f"Expected: {kaggle_json_src}"
    )

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
dest = os.path.join(kaggle_dir, 'kaggle.json')
shutil.copy(kaggle_json_src, dest)
try:
    os.chmod(dest, 0o600)
except Exception:
    pass
print(f"Kaggle credentials configured: {dest}")

# ── Download dataset ──────────────────────────────────────────────────────────
import kagglehub
print("Downloading monadarling143/stage-2-output-v2 ...")
DATASET_ROOT = kagglehub.dataset_download("monadarling143/stage-2-output-v2")
print(f"Dataset root: {DATASET_ROOT}")

# Walk and show structure
for root, dirs, files in os.walk(DATASET_ROOT):
    level = root.replace(DATASET_ROOT, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    parquets = [f for f in files if f.endswith('.parquet')]
    jsons    = [f for f in files if f.endswith('.json')]
    if parquets or jsons or level <= 2:
        print(f"{indent}{folder}/  "
              f"({len(parquets)} parquets, {len(jsons)} jsons)")

Kaggle credentials configured: /root/.kaggle/kaggle.json


100%|██████████| 1.66G/1.66G [01:02<00:00, 28.6MB/s]

Extracting files...


Dataset root: /root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1
1/  (0 parquets, 0 jsons)
  STAGE_2_OUTPUT/  (0 parquets, 0 jsons)
    stage_2_with_zero_v2/  (6 parquets, 3 jsons)
    stage_2_without_zero_v2/  (6 parquets, 3 jsons)


## Cell 3 — Locate Both Dataset Folders

In [16]:
import os

def find_dataset_dir(base_path, variant):
    # Walk base_path for folder containing stage2_X_train.parquet
    # variant = 'with' (not without) or 'without'
    
    for root, dirs, files in os.walk(base_path):
        if 'stage2_X_train.parquet' in files:
            root_lower = root.lower()
            if variant == 'with':
                # must contain 'with' but NOT 'without'
                if 'with' in root_lower and 'without' not in root_lower:
                    return root
            elif variant == 'without':
                if 'without' in root_lower:
                    return root
    return None

STAGE2_DIR_WITH    = find_dataset_dir(DATASET_ROOT, 'with')     # 167 features (VAE-A)
STAGE2_DIR_WITHOUT = find_dataset_dir(DATASET_ROOT, 'without')  # 140 features (VAE-B)

if STAGE2_DIR_WITH is None:
    raise FileNotFoundError(
        "Could not locate stage_2_with_zero_v2 folder.\n"
        f"Walked: {DATASET_ROOT}\n"
        "Ensure the Kaggle dataset contains stage_2_with_zero_v2/"
    )
if STAGE2_DIR_WITHOUT is None:
    raise FileNotFoundError(
        "Could not locate stage_2_without_zero_v2 folder.\n"
        f"Walked: {DATASET_ROOT}\n"
        "Ensure the Kaggle dataset contains stage_2_without_zero_v2/"
    )

print(f"WITH  (VAE-A, 167 features): {STAGE2_DIR_WITH}")
print(f"WITHOUT (VAE-B, 140 features): {STAGE2_DIR_WITHOUT}")

# Output directories
STAGE3_DIR_A = os.path.join(notebook_dir, 'vae_a_output')
STAGE3_DIR_B = os.path.join(notebook_dir, 'vae_b_output')
os.makedirs(STAGE3_DIR_A, exist_ok=True)
os.makedirs(STAGE3_DIR_B, exist_ok=True)

print(f"\nVAE-A output: {STAGE3_DIR_A}")
print(f"VAE-B output: {STAGE3_DIR_B}")

# Verify required files in both directories
REQUIRED = [
    'stage2_X_train.parquet', 'stage2_X_test.parquet',
    'stage2_sentinel_mask_train.parquet', 'stage2_sentinel_mask_test.parquet',
    'stage2_y_train.parquet', 'stage2_y_test.parquet',
    'stage2_feature_names.json', 'stage2_class_weights.json',
]

for label, d in [('WITH', STAGE2_DIR_WITH), ('WITHOUT', STAGE2_DIR_WITHOUT)]:
    print(f"\n{label} folder files:")
    for f in REQUIRED:
        path = os.path.join(d, f)
        if os.path.exists(path):
            print(f"  ok  {f} ({os.path.getsize(path)/1e6:.1f} MB)")
        else:
            raise FileNotFoundError(f"MISSING in {label}: {path}")

WITH  (VAE-A, 167 features): /root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_with_zero_v2
WITHOUT (VAE-B, 140 features): /root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_without_zero_v2

VAE-A output: /MINOR_PROJECT/vae_a_output
VAE-B output: /MINOR_PROJECT/vae_b_output

WITH folder files:
  ok  stage2_X_train.parquet (295.4 MB)
  ok  stage2_X_test.parquet (72.5 MB)
  ok  stage2_sentinel_mask_train.parquet (38.3 MB)
  ok  stage2_sentinel_mask_test.parquet (9.3 MB)
  ok  stage2_y_train.parquet (0.7 MB)
  ok  stage2_y_test.parquet (0.2 MB)
  ok  stage2_feature_names.json (0.0 MB)
  ok  stage2_class_weights.json (0.0 MB)

WITHOUT folder files:
  ok  stage2_X_train.parquet (295.2 MB)
  ok  stage2_X_test.parquet (72.4 MB)
  ok  stage2_sentinel_mask_train.parquet (32.1 MB)
  ok  stage2_sentinel_mask_test.parquet (7.8 MB)
  ok  stage2_y_train.parquet (0.7 MB)
  ok  stage2_y_test.parquet (0.2 M

## Cell 4 — Imports, Seeds & GPU Setup

In [17]:
import sys, json, time, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark     = True   # fastest conv algo
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32    = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_gpu_mem():
    if DEVICE.type == 'cuda':
        used  = torch.cuda.memory_allocated() / 1e6
        total = torch.cuda.get_device_properties(0).total_memory / 1e6
        return used, total
    return 0.0, 0.0

def log_gpu(label=''):
    used, total = get_gpu_mem()
    pct = 100 * used / total if total > 0 else 0
    print(f"  [{label:20s}] GPU: {used:6.0f}MB / {total:8.0f}MB ({pct:5.1f}%)")

def clear_cache():
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print(f"Python  : {sys.version}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    cc = torch.cuda.get_device_capability(0)
    print(f"Compute : {cc[0]}.{cc[1]}")
    print(f"TF32    : enabled (2x matrix multiply speedup)")
log_gpu('startup')

Python  : 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
PyTorch : 2.10.0+cu126
Device  : cuda
GPU     : NVIDIA GeForce RTX 4090
VRAM    : 25.4 GB
Compute : 8.9
TF32    : enabled (2x matrix multiply speedup)
  [startup             ] GPU:     38MB /    25394MB (  0.1%)


## Cell 5 — Shared Classes & Functions

In [18]:
# ── Dataset ──────────────────────────────────────────────────────────────────
def _read_parquet_chunked(path, out_dtype=np.float32):
    import pyarrow.parquet as pq
    pf     = pq.ParquetFile(path)
    chunks = []
    for i in range(pf.metadata.num_row_groups):
        tbl = pf.read_row_group(i)
        arr = np.column_stack([np.asarray(tbl.column(c), dtype=out_dtype)
                               for c in tbl.schema.names])
        chunks.append(arr)
    return np.concatenate(chunks, axis=0)


class SentinelAwareDataset(Dataset):
    def __init__(self, x_path, mask_path, label_path=None):
        print(f"  Loading {os.path.basename(x_path)} ...")
        X_np    = _read_parquet_chunked(x_path,    out_dtype=np.float16)
        mask_np = _read_parquet_chunked(mask_path, out_dtype=np.uint8)
        self.X    = torch.from_numpy(X_np)
        self.mask = torch.from_numpy(mask_np).bool()
        self.n_features = self.X.shape[1]
        if label_path:
            y_df  = pd.read_parquet(label_path)
            y_col = 'y' if 'y' in y_df.columns else y_df.columns[0]
            self.y = torch.tensor(y_df[y_col].values, dtype=torch.long)
        else:
            self.y = None
        ram = (self.X.element_size()*self.X.nelement() +
               self.mask.element_size()*self.mask.nelement()) / 1e6
        print(f"    shape={self.X.shape}  RAM~{ram:.0f}MB")
        if self.y is not None:
            labels, counts = torch.unique(self.y, return_counts=True)
            for l, c in zip(labels.tolist(), counts.tolist()):
                print(f"    class {l}: {c:>10,}")

    def __len__(self):  return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].float()
        if self.y is not None:
            return x, self.mask[idx], self.y[idx]
        return x, self.mask[idx]


# ── Model ─────────────────────────────────────────────────────────────────────
def reparameterise(mu, log_var):
    return mu + torch.exp(0.5 * log_var) * torch.randn_like(mu)

def to_quantum_angles(mu):
    # Inference only. Output in (0, pi).
    return torch.sigmoid(mu) * torch.pi


class SentinelAwareVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8, hidden_dims=None):
        super().__init__()
        self.input_dim  = input_dim
        self.latent_dim = latent_dim
        if hidden_dims is None:
            hidden_dims = [512, 256, 128]
        # Encoder
        enc, in_d = [], input_dim
        for h in hidden_dims:
            enc += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(0.05)]
            in_d = h
        self.encoder    = nn.Sequential(*enc)
        self.fc_mu      = nn.Linear(in_d, latent_dim)
        self.fc_log_var = nn.Linear(in_d, latent_dim)
        # Decoder
        dec, in_d = [], latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.GELU()]
            in_d = h
        dec += [nn.Linear(in_d, input_dim), nn.Sigmoid()]
        self.decoder = nn.Sequential(*dec)

    def encode(self, x):
        h  = self.encoder(x)
        mu = self.fc_mu(h)
        lv = torch.clamp(self.fc_log_var(h), -4, 4)
        return mu, lv

    def decode(self, z):   return self.decoder(z)

    def forward(self, x):
        mu, lv = self.encode(x)
        return self.decode(reparameterise(mu, lv)), mu, lv


# ── Loss ──────────────────────────────────────────────────────────────────────
def vae_loss(x, x_recon, mu, log_var, sentinel_mask,
             beta=1.0, free_bits=1.5,
             y_labels=None, aux_weight=0.7, aux_clf=None,
             class_weights_tensor=None):
    # Sentinel-masked reconstruction loss
    w          = (~sentinel_mask).float()
    sq_err     = w * (x - x_recon).pow(2)
    real_count = w.sum(dim=1).clamp(min=1)
    recon_loss = (sq_err.sum(dim=1) / real_count).mean()

    # KL — average batch first, then floor per-dim (correct aggregation order)
    kl_per_dim = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp())
    kl_per_dim = torch.clamp(kl_per_dim.mean(dim=0), min=free_bits)
    kl_loss    = kl_per_dim.sum()

    # Aux classification loss — class-weighted cross entropy
    aux_loss = torch.tensor(0.0, device=mu.device)
    if y_labels is not None and aux_clf is not None:
        logits = aux_clf(mu)
        if class_weights_tensor is not None:
            aux_loss = F.cross_entropy(logits, y_labels,
                                       weight=class_weights_tensor.to(mu.device))
        else:
            aux_loss = F.cross_entropy(logits, y_labels)

    return recon_loss + beta * kl_loss + aux_weight * aux_loss, recon_loss, kl_loss, aux_loss


class KLAnnealer:
    def __init__(self, beta_target=2.0, warmup_epochs=40):
        self.beta_target   = beta_target
        self.warmup_epochs = warmup_epochs
    def get_beta(self, epoch):
        return min(epoch / max(self.warmup_epochs, 1), 1.0) * self.beta_target


# ── Training loop ─────────────────────────────────────────────────────────────
def train_vae(model, train_loader, val_loader, config, device,
              stage3_dir, class_weights_tensor=None):
    mn = config['model_name']

    aux_clf   = nn.Linear(8, 5).to(device)
    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(aux_clf.parameters()),
        lr=config['lr'], weight_decay=config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        factor=config['scheduler_factor'],
        patience=config['scheduler_patience'],
        threshold=config.get('scheduler_threshold', 1e-4),
        threshold_mode='abs',
    )
    annealer = KLAnnealer(config['beta_target'], config['warmup_epochs'])
    scaler   = GradScaler() if config.get('use_amp') else None

    best_val_recon = float('inf')
    best_epoch     = 0
    patience_ctr   = 0
    history        = []
    ckpt_path      = os.path.join(stage3_dir, f'{mn}_best.pt')
    free_bits      = config.get('free_bits', 1.5)
    aux_weight     = config.get('aux_weight', 0.7)

    print(f"\n{'='*72}")
    print(f"  Training {mn.upper()}  |  {config['input_dim']} -> {config['hidden_dims']} -> 8")
    print(f"  max_epochs={config['max_epochs']}  patience={config['patience']}  "
          f"batch={config['batch_size']}")
    print(f"  beta: 0 -> {config['beta_target']} over {config['warmup_epochs']} epochs  |  "
          f"free_bits={free_bits}  aux_weight={aux_weight}")
    print(f"  AMP={config.get('use_amp',False)}  "
          f"class_weights={'YES (v2)' if class_weights_tensor is not None else 'NO'}")
    print(f"{'='*72}\n")

    t0 = time.time()

    for epoch in range(1, config['max_epochs'] + 1):
        beta = annealer.get_beta(epoch - 1)

        # Training
        model.train();  aux_clf.train()
        tr_tot = tr_r = tr_k = tr_a = 0.0
        nb = 0
        for batch in train_loader:
            x, mask, y_lbl = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            if config.get('use_amp'):
                with autocast(dtype=torch.float16):
                    xr, mu, lv = model(x)
                    loss, r, k, a = vae_loss(x, xr, mu, lv, mask, beta, free_bits,
                                             y_lbl, aux_weight, aux_clf,
                                             class_weights_tensor)
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) + list(aux_clf.parameters()),
                    config['grad_clip'])
                scaler.step(optimizer);  scaler.update()
            else:
                xr, mu, lv = model(x)
                loss, r, k, a = vae_loss(x, xr, mu, lv, mask, beta, free_bits,
                                         y_lbl, aux_weight, aux_clf,
                                         class_weights_tensor)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) + list(aux_clf.parameters()),
                    config['grad_clip'])
                optimizer.step()
            tr_tot += loss.item();  tr_r += r.item()
            tr_k   += k.item();     tr_a += a.item()
            nb += 1
        tr_tot /= nb;  tr_r /= nb;  tr_k /= nb;  tr_a /= nb

        # Validation — no aux loss
        model.eval();  aux_clf.eval()
        vl_tot = vl_r = vl_k = 0.0
        nv = 0
        with torch.no_grad():
            for batch in val_loader:
                x, mask = batch[0].to(device), batch[1].to(device)
                if config.get('use_amp'):
                    with autocast(dtype=torch.float16):
                        xr, mu, lv = model(x)
                        loss, r, k, _ = vae_loss(x, xr, mu, lv, mask,
                                                  beta, free_bits,
                                                  None, aux_weight, None)
                else:
                    xr, mu, lv = model(x)
                    loss, r, k, _ = vae_loss(x, xr, mu, lv, mask,
                                              beta, free_bits,
                                              None, aux_weight, None)
                vl_tot += loss.item();  vl_r += r.item();  vl_k += k.item()
                nv += 1
        vl_tot /= nv;  vl_r /= nv;  vl_k /= nv

        # LR scheduler
        prev_lr = optimizer.param_groups[0]['lr']
        scheduler.step(vl_r)
        cur_lr = optimizer.param_groups[0]['lr']
        if cur_lr < prev_lr:
            print(f"  [LR] {prev_lr:.2e} -> {cur_lr:.2e}")

        # Warmup boundary reset
        if epoch == config['warmup_epochs']:
            best_val_recon = float('inf')
            patience_ctr   = 0
            print(f"  [Warmup end E{epoch}] beta={beta:.3f} fixed. Tracker reset.")

        history.append({'epoch': epoch, 'beta': beta, 'lr': cur_lr,
                        'tr_recon': tr_r, 'tr_kl': tr_k, 'tr_aux': tr_a,
                        'val_recon': vl_r, 'val_kl': vl_k})

        if epoch % 5 == 0 or epoch == 1:
            used, total = get_gpu_mem()
            print(f"[E{epoch:03d}] b={beta:.2f} lr={cur_lr:.1e} | "
                  f"tr_r={tr_r:.4f} tr_k={tr_k:.3f} tr_a={tr_a:.4f} | "
                  f"val_r={vl_r:.4f} val_k={vl_k:.3f} | "
                  f"GPU {used:.0f}/{total:.0f}MB | {time.time()-t0:.0f}s")

        # Collapse detection
        if epoch > config['warmup_epochs'] and vl_k < 0.01:
            print(f"\n  COLLAPSE at E{epoch}  val_kl={vl_k:.6f} — stopping.")
            break

        # Early stopping + checkpoint
        if vl_r < best_val_recon - 1e-6:
            best_val_recon = vl_r
            best_epoch     = epoch
            patience_ctr   = 0
            if epoch >= config['warmup_epochs']:
                torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                            'aux_clf_state': aux_clf.state_dict(),
                            'optimizer_state': optimizer.state_dict(),
                            'val_recon': vl_r, 'val_kl': vl_k,
                            'config': config}, ckpt_path)
                print(f"  [CKP] E{epoch}  val_recon={vl_r:.6f}  val_kl={vl_k:.4f}")
        else:
            if epoch > config['warmup_epochs']:
                patience_ctr += 1
                if patience_ctr >= config['patience']:
                    print(f"\n  [EarlyStop] No improvement for {config['patience']} epochs.")
                    break

    elapsed = time.time() - t0
    print(f"\n  [{mn}] Done in {elapsed:.0f}s ({elapsed/60:.1f} min) | "
          f"best E{best_epoch}  val_recon={best_val_recon:.6f}")
    return history, aux_clf, best_epoch, best_val_recon


# ── Latent extraction ─────────────────────────────────────────────────────────
def extract_latent(model, loader, device, out_path):
    model.eval()
    all_angles = []
    with torch.no_grad():
        for batch in loader:
            x = batch[0].to(device)
            mu, _ = model.encode(x)
            all_angles.append(to_quantum_angles(mu).cpu())
    z = torch.cat(all_angles, dim=0)
    assert z.shape[1] == 8
    assert (z >= 0).all() and (z <= torch.pi).all()
    assert not torch.isnan(z).any() and not torch.isinf(z).any()
    pd.DataFrame(z.numpy(), columns=[f'z_{i}' for i in range(8)]) \
      .to_parquet(out_path, index=False)
    print(f"  Saved {out_path}  shape={z.shape}")
    print(f"  range=[{z.min():.4f}, {z.max():.4f}]")
    print(f"  per-dim std: {[f'{s:.3f}' for s in z.std(dim=0).tolist()]}")
    return z


# ── Validation gate ───────────────────────────────────────────────────────────
def validate_outputs(z_train, z_test, y_train, y_test, name):
    CN = {0:'NORMAL', 1:'DoSDDoS', 2:'PROBE', 3:'EXPLOIT', 4:'MALWARE'}
    print(f"\n{'='*58}\n  8-POINT GATE: {name}\n{'='*58}")
    # 1
    assert z_train.shape[1] == 8 == z_test.shape[1]
    assert z_train.shape[0] == len(y_train) and z_test.shape[0] == len(y_test)
    print("  [1/8] Shape consistency:          PASS")
    # 2
    eps = 1e-6
    assert float(z_train.min()) >= -eps and float(z_train.max()) <= torch.pi + eps
    assert float(z_test.min())  >= -eps and float(z_test.max())  <= torch.pi + eps
    print("  [2/8] Range [0, pi]:              PASS")
    # 3
    assert not torch.isnan(z_train).any() and not torch.isnan(z_test).any()
    print("  [3/8] No NaN:                     PASS")
    # 4
    assert not torch.isinf(z_train).any() and not torch.isinf(z_test).any()
    print("  [4/8] No Inf:                     PASS")
    # 5
    dim_std = z_train.std(dim=0)
    coll    = (dim_std < 0.05).nonzero(as_tuple=True)[0].tolist()
    print(f"  [5/8] Angle diversity:            "
          f"{'PASS' if not coll else f'WARN {coll}'}  "
          f"(min={dim_std.min():.4f}, target>=0.60)")
    # 6
    classes   = torch.unique(y_train)
    centroids = {int(c): z_train[y_train == c].mean(0) for c in classes}
    min_d, min_pair = float('inf'), ('?','?')
    for i in range(len(classes)):
        for j in range(i+1, len(classes)):
            ci, cj = int(classes[i]), int(classes[j])
            d = (centroids[ci] - centroids[cj]).norm().item()
            if d < min_d: min_d, min_pair = d, (CN[ci], CN[cj])
    print(f"  [6/8] Min class distance:         {min_d:.4f}  "
          f"({'PASS' if min_d >= 1.5 else 'WARN<1.5'})  "
          f"pair={min_pair[0]} vs {min_pair[1]}")
    # 7
    tst_std  = z_test.std(dim=0)
    const_t  = (tst_std < 1e-4).nonzero(as_tuple=True)[0].tolist()
    print(f"  [7/8] Test angle diversity:       "
          f"{'PASS' if not const_t else f'WARN {const_t}'}")
    # 8
    assert set(torch.unique(y_test).tolist()) == {0,1,2,3,4}
    print("  [8/8] All 5 classes in test set:  PASS")
    print(f"\n  val_kl target 8–14  |  class_dist target 1.5–2.2  |  std_min target 0.60")
    print(f"{'='*58}")
    return {'dim_std': dim_std.tolist(), 'min_class_dist': min_d,
            'min_class_pair': list(min_pair), 'collapsed_dims': coll,
            'centroids': {CN[k]: v.tolist() for k, v in centroids.items()}}


# ── Training curves plot ──────────────────────────────────────────────────────
def plot_curves(history, name, stage3_dir):
    import matplotlib.pyplot as plt
    epochs   = [h['epoch']     for h in history]
    tr_r     = [h['tr_recon']  for h in history]
    vl_r     = [h['val_recon'] for h in history]
    tr_k     = [h['tr_kl']     for h in history]
    vl_k     = [h['val_kl']    for h in history]
    tr_a     = [h['tr_aux']    for h in history]
    betas    = [h['beta']      for h in history]

    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    axes[0].plot(epochs, tr_r, label='Train', color='tab:blue')
    axes[0].plot(epochs, vl_r, label='Val',   color='tab:orange')
    axes[0].axhline(0.0007, color='green', ls='--', alpha=0.6, label='Target max')
    axes[0].set_title('Reconstruction Loss');  axes[0].legend();  axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, tr_k, label='Train', color='tab:blue')
    axes[1].plot(epochs, vl_k, label='Val',   color='tab:orange')
    axes[1].axhline(8.0,  color='green', ls='--', alpha=0.6, label='Min target')
    axes[1].axhline(14.0, color='red',   ls='--', alpha=0.6, label='Max target')
    axes[1].set_title('KL Divergence');  axes[1].legend();  axes[1].grid(alpha=0.3)
    axes[2].plot(epochs, tr_a, color='tab:purple')
    axes[2].set_title('Aux Loss (class-weighted)');  axes[2].grid(alpha=0.3)
    axes[3].plot(epochs, betas, color='tab:green')
    axes[3].set_title('Beta Annealing Schedule');    axes[3].grid(alpha=0.3)
    fig.suptitle(f'{name} Training Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(stage3_dir, f'{name.lower()}_training_curves.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved {out}")


# ── Angle distribution plot ───────────────────────────────────────────────────
def plot_angles(z_train, y_tensor, name, stage3_dir):
    CN = {0:'NORMAL', 1:'DoSDDoS', 2:'PROBE', 3:'EXPLOIT', 4:'MALWARE'}
    CC = {0:'#2196F3', 1:'#F44336', 2:'#FF9800', 3:'#9C27B0', 4:'#4CAF50'}
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes = axes.flatten()
    for dim in range(8):
        ax = axes[dim]
        for cls_id in range(5):
            vals = z_train[y_tensor == cls_id, dim].numpy()
            ax.hist(vals, bins=50, alpha=0.5, label=CN[cls_id],
                    color=CC[cls_id], density=True)
        ax.set_title(f'z_{dim}  std={z_train[:, dim].std():.3f}')
        ax.axvline(x=np.pi/2, color='gray', ls='--', alpha=0.5)
        if dim == 0: ax.legend(fontsize=7)
    fig.suptitle(f'{name} Angle Distributions by Class', fontsize=13, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(stage3_dir, f'{name.lower()}_angle_distributions.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved {out}")


# ── t-SNE plot ────────────────────────────────────────────────────────────────
def plot_tsne(z_train, y_tensor, name, stage3_dir):
    from sklearn.manifold import TSNE
    CN = {0:'NORMAL', 1:'DoSDDoS', 2:'PROBE', 3:'EXPLOIT', 4:'MALWARE'}
    CC = {0:'#2196F3', 1:'#F44336', 2:'#FF9800', 3:'#9C27B0', 4:'#4CAF50'}
    N = 5000
    idx = torch.cat([
        (y_tensor == c).nonzero(as_tuple=True)[0][
            torch.randperm((y_tensor == c).sum())[:N//5]
        ] for c in range(5)
    ])
    z_s = z_train[idx].numpy()
    y_s = y_tensor[idx].numpy()
    print(f"  Running t-SNE on {len(z_s)} samples ...")
    z2d = TSNE(n_components=2, perplexity=30, max_iter=1000,
               random_state=RANDOM_STATE).fit_transform(z_s)
    fig, ax = plt.subplots(figsize=(9, 7))
    for c in range(5):
        m = y_s == c
        ax.scatter(z2d[m,0], z2d[m,1], c=CC[c], label=CN[c], s=8, alpha=0.6)
    ax.set_title(f'{name}: t-SNE of 8D Latent Space (5K stratified)', fontsize=12)
    ax.legend(markerscale=3);  ax.grid(alpha=0.2)
    plt.tight_layout()
    out = os.path.join(stage3_dir, f'{name.lower()}_tsne.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved {out}")


# ── Per-class reconstruction loss ─────────────────────────────────────────────
def per_class_recon(model, val_loader, device, name):
    CN = {0:'NORMAL', 1:'DoSDDoS', 2:'PROBE', 3:'EXPLOIT', 4:'MALWARE'}
    model.eval()
    cls_losses = {i: [] for i in range(5)}
    with torch.no_grad():
        for batch in val_loader:
            x, mask, y = batch[0].to(device), batch[1].to(device), batch[2]
            xr, mu, lv = model(x)
            w  = (~mask).float()
            sq = w * (x - xr).pow(2)
            pl = (sq.sum(1) / w.sum(1).clamp(min=1)).cpu()
            for c in range(5):
                m = (y == c)
                if m.any(): cls_losses[c].append(pl[m])
    print(f"\n  Per-class reconstruction loss ({name}):")
    summary = {}
    for c in range(5):
        ls = torch.cat(cls_losses[c])
        m  = ls.mean().item()
        summary[CN[c]] = m
        print(f"    {CN[c]:>12s}: {m:.6f}  n={len(ls):>7,}")
    normal = summary['NORMAL']
    for cls in ['EXPLOIT', 'MALWARE']:
        r = summary[cls] / normal if normal > 0 else 0
        tag = f"  (SMOTE degradation {r:.1f}x)" if r > 2 else ""
        print(f"    {cls} / NORMAL ratio: {r:.2f}{tag}")
    return summary


# ── Save artefacts ────────────────────────────────────────────────────────────
def save_artefacts(name, stage3_dir, config, checkpoint, history,
                   z_train, z_test, val_results, feature_names,
                   class_weights_dict):
    ts = __import__('datetime').datetime.utcnow().isoformat()

    cfg_save = {**config,
                'best_epoch': checkpoint['epoch'],
                'best_val_recon': checkpoint['val_recon'],
                'best_val_kl':    checkpoint['val_kl'],
                'epochs_trained': len(history),
                'feature_names':  feature_names,
                'device': str(DEVICE)}
    with open(os.path.join(stage3_dir, f'{name}_config.json'), 'w') as f:
        json.dump(cfg_save, f, indent=2)

    CN = {0:'NORMAL', 1:'DoSDDoS', 2:'PROBE', 3:'EXPLOIT', 4:'MALWARE'}
    stats = {
        'model_name':       name.upper(),
        'input_dim':        config['input_dim'],
        'latent_dim':       8,
        'smote_version':    'v2',
        'class_weights_used': class_weights_dict,
        'per_dim_mean':     z_train.mean(0).tolist(),
        'per_dim_std':      z_train.std(0).tolist(),
        'per_dim_min':      z_train.min(0).values.tolist(),
        'per_dim_max':      z_train.max(0).values.tolist(),
        'class_centroids':  val_results['centroids'],
        'min_class_dist':   val_results['min_class_dist'],
        'min_class_pair':   val_results['min_class_pair'],
        'collapsed_dims':   val_results['collapsed_dims'],
        'z_train_shape':    list(z_train.shape),
        'z_test_shape':     list(z_test.shape),
        'timestamp':        ts,
    }
    with open(os.path.join(stage3_dir, f'{name}_latent_stats.json'), 'w') as f:
        json.dump(stats, f, indent=2)

    with open(os.path.join(stage3_dir, f'{name}_training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    expected = [f'{name}_best.pt', f'{name}_z_train.parquet',
                f'{name}_z_test.parquet',  f'{name}_config.json',
                f'{name}_latent_stats.json', f'{name}_training_history.json',
                f'{name.lower()}_training_curves.png',
                f'{name.lower()}_angle_distributions.png',
                f'{name.lower()}_tsne.png']
    print(f"\n  Artefacts in {os.path.basename(stage3_dir)}/:")
    all_ok = True
    for fn in expected:
        fp = os.path.join(stage3_dir, fn)
        ok = os.path.exists(fp)
        if not ok: all_ok = False
        sz = os.path.getsize(fp)/1e6 if ok else -1
        print(f"    {'ok' if ok else 'MISS'}  {fn}  ({sz:.1f} MB)")
    print(f"  {'ALL ARTEFACTS SAVED' if all_ok else 'WARNING: SOME MISSING'}")
    return all_ok


print("All shared classes and functions defined.")

All shared classes and functions defined.


## Cell 6 — Load Class Weights & Build CONFIG Templates

In [19]:
# Load class weights from Stage 2 v2 — same weights for both models
# (same y_train_smote used for both)
with open(os.path.join(STAGE2_DIR_WITHOUT, 'stage2_class_weights.json')) as f:
    cw_data = json.load(f)

CLASS_WEIGHTS_DICT = {int(k): float(v)
                      for k, v in cw_data['class_weights'].items()}

CLASS_WEIGHTS_TENSOR = torch.tensor(
    [CLASS_WEIGHTS_DICT[i] for i in range(5)], dtype=torch.float32
)

print("Class weights loaded from stage2_class_weights.json:")
cw_names = ['Normal', 'DoS/DDoS', 'Recon', 'Exploit', 'Malware']
for i, (name, w) in enumerate(zip(cw_names, CLASS_WEIGHTS_TENSOR.tolist())):
    bar = '█' * int(w / 2)
    print(f"  Class {i} {name:<12}: {w:>8.4f}  {bar}")

# ── Shared hyperparameters — same for both models (fair ablation) ─────────────
BATCH_SIZE_TRAIN = 2048
BATCH_SIZE_VAL   = 4096
NUM_WORKERS      = 4
PIN_MEMORY       = DEVICE.type == 'cuda'
PERSISTENT_W     = (NUM_WORKERS > 0)

BASE_CONFIG = {
    'latent_dim':          8,
    'hidden_dims':         [512, 256, 128],
    'lr':                  1e-3,
    'weight_decay':        1e-4,
    'batch_size':          BATCH_SIZE_TRAIN,
    'max_epochs':          120,
    'patience':            25,
    'beta_target':         2.0,
    'warmup_epochs':       40,
    'free_bits':           2,    # 8 × 1.5 = 12.0 nats KL floor
    'aux_weight':          0.7,    # class-weighted aux loss
    'grad_clip':           1.0,
    'scheduler_factor':    0.5,
    'scheduler_patience':  25,     # must be >= warmup_epochs
    'scheduler_threshold': 1e-4,   # prevents E001 anomaly LR cut
    'use_amp':             True,
    'amp_dtype':           'float16',
}

print(f"\nBase CONFIG prepared (shared between VAE-A and VAE-B)")
print(f"  beta_target={BASE_CONFIG['beta_target']}  "
      f"warmup={BASE_CONFIG['warmup_epochs']}  "
      f"free_bits={BASE_CONFIG['free_bits']}  "
      f"aux_weight={BASE_CONFIG['aux_weight']}")

Class weights loaded from stage2_class_weights.json:
  Class 0 Normal      :   0.2626  
  Class 1 DoS/DDoS    :   1.5220  
  Class 2 Recon       :   3.1747  █
  Class 3 Exploit     :   5.0127  ██
  Class 4 Malware     :  47.6208  ███████████████████████

Base CONFIG prepared (shared between VAE-A and VAE-B)
  beta_target=2.0  warmup=40  free_bits=2  aux_weight=0.7


## Cell 7 — Train VAE-B (WITHOUT near-zero, 140 features)
**Output → `vae_b_output/`**

In [20]:
# ── Load metadata ─────────────────────────────────────────────────────────────
with open(os.path.join(STAGE2_DIR_WITHOUT, 'stage2_feature_names.json')) as f:
    s2_b = json.load(f)
INPUT_DIM_B  = s2_b['n_features']   # 140
QUANTUM_DIM  = s2_b['quantum_dim']  # 8
FEAT_NAMES_B = s2_b['feature_names']
assert INPUT_DIM_B == 140, f"Expected 140, got {INPUT_DIM_B}"
print(f"VAE-B: INPUT_DIM={INPUT_DIM_B}  QUANTUM_DIM={QUANTUM_DIM}")

# ── Datasets ──────────────────────────────────────────────────────────────────
print("\nLoading WITHOUT datasets:")
ds_b_train = SentinelAwareDataset(
    os.path.join(STAGE2_DIR_WITHOUT, 'stage2_X_train.parquet'),
    os.path.join(STAGE2_DIR_WITHOUT, 'stage2_sentinel_mask_train.parquet'),
    os.path.join(STAGE2_DIR_WITHOUT, 'stage2_y_train.parquet'),
)
ds_b_test = SentinelAwareDataset(
    os.path.join(STAGE2_DIR_WITHOUT, 'stage2_X_test.parquet'),
    os.path.join(STAGE2_DIR_WITHOUT, 'stage2_sentinel_mask_test.parquet'),
    os.path.join(STAGE2_DIR_WITHOUT, 'stage2_y_test.parquet'),
)
assert ds_b_train.n_features == INPUT_DIM_B
assert ds_b_test.n_features  == INPUT_DIM_B

loader_b_train = DataLoader(ds_b_train, batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,  num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_W)
loader_b_val   = DataLoader(ds_b_test,  batch_size=BATCH_SIZE_VAL,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_W)

print(f"\nTrain batches: {len(loader_b_train)}  Val batches: {len(loader_b_val)}")
log_gpu('B datasets loaded')

# ── Config & model ────────────────────────────────────────────────────────────
CONFIG_B = {**BASE_CONFIG, 'model_name': 'vae_b', 'input_dim': INPUT_DIM_B}
model_b  = SentinelAwareVAE(INPUT_DIM_B, CONFIG_B['latent_dim'],
                             CONFIG_B['hidden_dims']).to(DEVICE)
print(f"\nVAE-B params: {sum(p.numel() for p in model_b.parameters()):,}")
log_gpu('B model init')

# ── Train ─────────────────────────────────────────────────────────────────────
clear_cache()
history_b, aux_clf_b, best_ep_b, best_recon_b = train_vae(
    model_b, loader_b_train, loader_b_val, CONFIG_B, DEVICE,
    STAGE3_DIR_B, CLASS_WEIGHTS_TENSOR
)

# Save history
with open(os.path.join(STAGE3_DIR_B, 'vae_b_training_history.json'), 'w') as f:
    json.dump(history_b, f, indent=2)
log_gpu('B training complete')
clear_cache()

VAE-B: INPUT_DIM=140  QUANTUM_DIM=8

Loading WITHOUT datasets:
  Loading stage2_X_train.parquet ...
    shape=torch.Size([2381042, 140])  RAM~1000MB
    class 0:  1,813,161
    class 1:    312,881
    class 2:    150,000
    class 3:     95,000
    class 4:     10,000
  Loading stage2_X_test.parquet ...
    shape=torch.Size([573807, 140])  RAM~241MB
    class 0:    453,290
    class 1:     78,221
    class 2:     29,137
    class 3:     11,966
    class 4:      1,193

Train batches: 1163  Val batches: 141
  [B datasets loaded   ] GPU:     38MB /    25394MB (  0.1%)

VAE-B params: 479,644
  [B model init        ] GPU:     36MB /    25394MB (  0.1%)

  Training VAE_B  |  140 -> [512, 256, 128] -> 8
  max_epochs=120  patience=25  batch=2048
  beta: 0 -> 2.0 over 40 epochs  |  free_bits=2  aux_weight=0.7
  AMP=True  class_weights=YES (v2)



/tmp/ipykernel_1574/2271207170.py:145: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler   = GradScaler() if config.get('use_amp') else None
/tmp/ipykernel_1574/2271207170.py:177: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
/tmp/ipykernel_1574/2271207170.py:213: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


[E001] b=0.00 lr=1.0e-03 | tr_r=0.0037 tr_k=56.403 tr_a=0.4917 | val_r=0.0009 val_k=72.784 | GPU 42/25394MB | 16s
[E005] b=0.20 lr=1.0e-03 | tr_r=0.0019 tr_k=16.000 tr_a=0.3957 | val_r=0.0015 val_k=16.000 | GPU 42/25394MB | 91s
[E010] b=0.45 lr=1.0e-03 | tr_r=0.0013 tr_k=16.000 tr_a=0.3716 | val_r=0.0010 val_k=16.000 | GPU 42/25394MB | 176s
[E015] b=0.70 lr=1.0e-03 | tr_r=0.0011 tr_k=16.000 tr_a=0.3589 | val_r=0.0007 val_k=16.000 | GPU 42/25394MB | 259s
[E020] b=0.95 lr=1.0e-03 | tr_r=0.0010 tr_k=16.000 tr_a=0.3509 | val_r=0.0006 val_k=16.000 | GPU 42/25394MB | 350s
[E025] b=1.20 lr=1.0e-03 | tr_r=0.0010 tr_k=16.000 tr_a=0.3456 | val_r=0.0006 val_k=16.000 | GPU 42/25394MB | 443s
[E030] b=1.45 lr=1.0e-03 | tr_r=0.0008 tr_k=16.000 tr_a=0.3400 | val_r=0.0005 val_k=16.000 | GPU 42/25394MB | 529s
[E035] b=1.70 lr=1.0e-03 | tr_r=0.0008 tr_k=16.000 tr_a=0.3370 | val_r=0.0005 val_k=16.000 | GPU 42/25394MB | 614s
  [Warmup end E40] beta=1.950 fixed. Tracker reset.
[E040] b=1.95 lr=1.0e-03 | tr_

## Cell 8 — VAE-B: Load Checkpoint, Extract Latents, Validate, Plot

In [21]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt_b = torch.load(os.path.join(STAGE3_DIR_B, 'vae_b_best.pt'), map_location=DEVICE)
model_b.load_state_dict(ckpt_b['model_state'])
model_b.eval()
print(f"VAE-B best checkpoint: E{ckpt_b['epoch']}  "
      f"val_recon={ckpt_b['val_recon']:.6f}  val_kl={ckpt_b['val_kl']:.4f}")

# ── Extract latent vectors ────────────────────────────────────────────────────
loader_b_extr_tr = DataLoader(ds_b_train, batch_size=BATCH_SIZE_VAL,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
loader_b_extr_te = DataLoader(ds_b_test,  batch_size=BATCH_SIZE_VAL,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("\nExtracting z_train (VAE-B) ...")
z_b_train = extract_latent(model_b, loader_b_extr_tr, DEVICE,
                            os.path.join(STAGE3_DIR_B, 'vae_b_z_train.parquet'))
print("\nExtracting z_test (VAE-B) ...")
z_b_test  = extract_latent(model_b, loader_b_extr_te, DEVICE,
                            os.path.join(STAGE3_DIR_B, 'vae_b_z_test.parquet'))
clear_cache()

# ── Validate ──────────────────────────────────────────────────────────────────
y_b_train = ds_b_train.y
y_b_test  = ds_b_test.y
val_results_b = validate_outputs(z_b_train, z_b_test, y_b_train, y_b_test, 'VAE-B')

# ── Plots ─────────────────────────────────────────────────────────────────────
plot_curves(history_b, 'VAE-B', STAGE3_DIR_B)
plot_angles(z_b_train, y_b_train, 'VAE-B', STAGE3_DIR_B)
plot_tsne(z_b_train, y_b_train, 'VAE-B', STAGE3_DIR_B)

# ── Per-class reconstruction loss ─────────────────────────────────────────────
pcl_b = per_class_recon(model_b, loader_b_val, DEVICE, 'VAE-B')

# ── Save artefacts ────────────────────────────────────────────────────────────
save_artefacts('vae_b', STAGE3_DIR_B, CONFIG_B, ckpt_b, history_b,
               z_b_train, z_b_test, val_results_b, FEAT_NAMES_B,
               CLASS_WEIGHTS_DICT)

VAE-B best checkpoint: E111  val_recon=0.000350  val_kl=16.0000

Extracting z_train (VAE-B) ...
  Saved /MINOR_PROJECT/vae_b_output/vae_b_z_train.parquet  shape=torch.Size([2381042, 8])
  range=[0.0000, 3.1416]
  per-dim std: ['0.620', '0.732', '0.673', '0.717', '0.743', '0.679', '0.679', '0.612']

Extracting z_test (VAE-B) ...
  Saved /MINOR_PROJECT/vae_b_output/vae_b_z_test.parquet  shape=torch.Size([573807, 8])
  range=[0.0000, 3.1416]
  per-dim std: ['0.585', '0.737', '0.664', '0.706', '0.724', '0.679', '0.675', '0.555']

  8-POINT GATE: VAE-B
  [1/8] Shape consistency:          PASS
  [2/8] Range [0, pi]:              PASS
  [3/8] No NaN:                     PASS
  [4/8] No Inf:                     PASS
  [5/8] Angle diversity:            PASS  (min=0.6117, target>=0.60)
  [6/8] Min class distance:         2.1685  (PASS)  pair=EXPLOIT vs MALWARE
  [7/8] Test angle diversity:       PASS
  [8/8] All 5 classes in test set:  PASS

  val_kl target 8–14  |  class_dist target 1.5–2.2  | 

/tmp/ipykernel_1574/2271207170.py:473: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = __import__('datetime').datetime.utcnow().isoformat()



  Artefacts in vae_b_output/:
    ok  vae_b_best.pt  (5.8 MB)
    ok  vae_b_z_train.parquet  (87.9 MB)
    ok  vae_b_z_test.parquet  (22.9 MB)
    ok  vae_b_config.json  (0.0 MB)
    ok  vae_b_latent_stats.json  (0.0 MB)
    ok  vae_b_training_history.json  (0.0 MB)
    MISS  vae_b_training_curves.png  (-1.0 MB)
    MISS  vae_b_angle_distributions.png  (-1.0 MB)
    MISS  vae_b_tsne.png  (-1.0 MB)


False

## Cell 9 — Train VAE-A (WITH near-zero, 167 features)
**Output → `vae_a_output/`**

In [22]:
# ── Load metadata ─────────────────────────────────────────────────────────────
with open(os.path.join(STAGE2_DIR_WITH, 'stage2_feature_names.json')) as f:
    s2_a = json.load(f)
INPUT_DIM_A  = s2_a['n_features']   # 167
FEAT_NAMES_A = s2_a['feature_names']
assert INPUT_DIM_A == 167, f"Expected 167, got {INPUT_DIM_A}"
print(f"VAE-A: INPUT_DIM={INPUT_DIM_A}  QUANTUM_DIM={QUANTUM_DIM}")

# ── Datasets ──────────────────────────────────────────────────────────────────
print("\nLoading WITH datasets:")
ds_a_train = SentinelAwareDataset(
    os.path.join(STAGE2_DIR_WITH, 'stage2_X_train.parquet'),
    os.path.join(STAGE2_DIR_WITH, 'stage2_sentinel_mask_train.parquet'),
    os.path.join(STAGE2_DIR_WITH, 'stage2_y_train.parquet'),
)
ds_a_test = SentinelAwareDataset(
    os.path.join(STAGE2_DIR_WITH, 'stage2_X_test.parquet'),
    os.path.join(STAGE2_DIR_WITH, 'stage2_sentinel_mask_test.parquet'),
    os.path.join(STAGE2_DIR_WITH, 'stage2_y_test.parquet'),
)
assert ds_a_train.n_features == INPUT_DIM_A
assert ds_a_test.n_features  == INPUT_DIM_A

loader_a_train = DataLoader(ds_a_train, batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,  num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_W)
loader_a_val   = DataLoader(ds_a_test,  batch_size=BATCH_SIZE_VAL,
    shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_W)

print(f"\nTrain batches: {len(loader_a_train)}  Val batches: {len(loader_a_val)}")
log_gpu('A datasets loaded')

# ── Config & model ────────────────────────────────────────────────────────────
CONFIG_A = {**BASE_CONFIG, 'model_name': 'vae_a', 'input_dim': INPUT_DIM_A}
model_a  = SentinelAwareVAE(INPUT_DIM_A, CONFIG_A['latent_dim'],
                             CONFIG_A['hidden_dims']).to(DEVICE)
print(f"\nVAE-A params: {sum(p.numel() for p in model_a.parameters()):,}")
log_gpu('A model init')

# ── Train ─────────────────────────────────────────────────────────────────────
clear_cache()
history_a, aux_clf_a, best_ep_a, best_recon_a = train_vae(
    model_a, loader_a_train, loader_a_val, CONFIG_A, DEVICE,
    STAGE3_DIR_A, CLASS_WEIGHTS_TENSOR
)

with open(os.path.join(STAGE3_DIR_A, 'vae_a_training_history.json'), 'w') as f:
    json.dump(history_a, f, indent=2)
log_gpu('A training complete')
clear_cache()

VAE-A: INPUT_DIM=167  QUANTUM_DIM=8

Loading WITH datasets:
  Loading stage2_X_train.parquet ...
    shape=torch.Size([2381042, 167])  RAM~1193MB
    class 0:  1,813,161
    class 1:    312,881
    class 2:    150,000
    class 3:     95,000
    class 4:     10,000
  Loading stage2_X_test.parquet ...
    shape=torch.Size([573807, 167])  RAM~287MB
    class 0:    453,290
    class 1:     78,221
    class 2:     29,137
    class 3:     11,966
    class 4:      1,193

Train batches: 1163  Val batches: 141
  [A datasets loaded   ] GPU:     38MB /    25394MB (  0.1%)

VAE-A params: 507,319
  [A model init        ] GPU:     36MB /    25394MB (  0.1%)

  Training VAE_A  |  167 -> [512, 256, 128] -> 8
  max_epochs=120  patience=25  batch=2048
  beta: 0 -> 2.0 over 40 epochs  |  free_bits=2  aux_weight=0.7
  AMP=True  class_weights=YES (v2)



/tmp/ipykernel_1574/2271207170.py:145: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler   = GradScaler() if config.get('use_amp') else None
/tmp/ipykernel_1574/2271207170.py:177: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
/tmp/ipykernel_1574/2271207170.py:213: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


[E001] b=0.00 lr=1.0e-03 | tr_r=0.0037 tr_k=56.437 tr_a=0.4912 | val_r=0.0008 val_k=77.088 | GPU 42/25394MB | 19s
[E005] b=0.20 lr=1.0e-03 | tr_r=0.0018 tr_k=16.000 tr_a=0.3951 | val_r=0.0012 val_k=16.004 | GPU 42/25394MB | 87s
[E010] b=0.45 lr=1.0e-03 | tr_r=0.0013 tr_k=16.000 tr_a=0.3714 | val_r=0.0009 val_k=16.000 | GPU 42/25394MB | 178s
[E015] b=0.70 lr=1.0e-03 | tr_r=0.0011 tr_k=16.000 tr_a=0.3578 | val_r=0.0007 val_k=16.005 | GPU 42/25394MB | 266s
[E020] b=0.95 lr=1.0e-03 | tr_r=0.0010 tr_k=16.000 tr_a=0.3504 | val_r=0.0006 val_k=16.000 | GPU 42/25394MB | 353s
[E025] b=1.20 lr=1.0e-03 | tr_r=0.0009 tr_k=16.000 tr_a=0.3440 | val_r=0.0006 val_k=16.000 | GPU 42/25394MB | 439s
[E030] b=1.45 lr=1.0e-03 | tr_r=0.0008 tr_k=16.000 tr_a=0.3399 | val_r=0.0005 val_k=16.000 | GPU 42/25394MB | 527s
[E035] b=1.70 lr=1.0e-03 | tr_r=0.0009 tr_k=16.000 tr_a=0.3355 | val_r=0.0005 val_k=16.000 | GPU 42/25394MB | 612s
  [Warmup end E40] beta=1.950 fixed. Tracker reset.
[E040] b=1.95 lr=1.0e-03 | tr_

## Cell 10 — VAE-A: Load Checkpoint, Extract Latents, Validate, Plot

In [23]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt_a = torch.load(os.path.join(STAGE3_DIR_A, 'vae_a_best.pt'), map_location=DEVICE)
model_a.load_state_dict(ckpt_a['model_state'])
model_a.eval()
print(f"VAE-A best checkpoint: E{ckpt_a['epoch']}  "
      f"val_recon={ckpt_a['val_recon']:.6f}  val_kl={ckpt_a['val_kl']:.4f}")

# ── Extract latent vectors ────────────────────────────────────────────────────
loader_a_extr_tr = DataLoader(ds_a_train, batch_size=BATCH_SIZE_VAL,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
loader_a_extr_te = DataLoader(ds_a_test,  batch_size=BATCH_SIZE_VAL,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("\nExtracting z_train (VAE-A) ...")
z_a_train = extract_latent(model_a, loader_a_extr_tr, DEVICE,
                            os.path.join(STAGE3_DIR_A, 'vae_a_z_train.parquet'))
print("\nExtracting z_test (VAE-A) ...")
z_a_test  = extract_latent(model_a, loader_a_extr_te, DEVICE,
                            os.path.join(STAGE3_DIR_A, 'vae_a_z_test.parquet'))
clear_cache()

# ── Validate ──────────────────────────────────────────────────────────────────
y_a_train = ds_a_train.y
y_a_test  = ds_a_test.y
val_results_a = validate_outputs(z_a_train, z_a_test, y_a_train, y_a_test, 'VAE-A')

# ── Plots ─────────────────────────────────────────────────────────────────────
plot_curves(history_a, 'VAE-A', STAGE3_DIR_A)
plot_angles(z_a_train, y_a_train, 'VAE-A', STAGE3_DIR_A)
plot_tsne(z_a_train, y_a_train, 'VAE-A', STAGE3_DIR_A)

# ── Per-class reconstruction loss ─────────────────────────────────────────────
pcl_a = per_class_recon(model_a, loader_a_val, DEVICE, 'VAE-A')

# ── Save artefacts ────────────────────────────────────────────────────────────
save_artefacts('vae_a', STAGE3_DIR_A, CONFIG_A, ckpt_a, history_a,
               z_a_train, z_a_test, val_results_a, FEAT_NAMES_A,
               CLASS_WEIGHTS_DICT)

VAE-A best checkpoint: E82  val_recon=0.000333  val_kl=16.0000

Extracting z_train (VAE-A) ...
  Saved /MINOR_PROJECT/vae_a_output/vae_a_z_train.parquet  shape=torch.Size([2381042, 8])
  range=[0.0000, 3.1416]
  per-dim std: ['0.738', '0.675', '0.686', '0.757', '0.679', '0.686', '0.658', '0.740']

Extracting z_test (VAE-A) ...
  Saved /MINOR_PROJECT/vae_a_output/vae_a_z_test.parquet  shape=torch.Size([573807, 8])
  range=[0.0000, 3.1416]
  per-dim std: ['0.729', '0.667', '0.660', '0.761', '0.671', '0.663', '0.656', '0.693']

  8-POINT GATE: VAE-A
  [1/8] Shape consistency:          PASS
  [2/8] Range [0, pi]:              PASS
  [3/8] No NaN:                     PASS
  [4/8] No Inf:                     PASS
  [5/8] Angle diversity:            PASS  (min=0.6584, target>=0.60)
  [6/8] Min class distance:         2.4497  (PASS)  pair=EXPLOIT vs MALWARE
  [7/8] Test angle diversity:       PASS
  [8/8] All 5 classes in test set:  PASS

  val_kl target 8–14  |  class_dist target 1.5–2.2  |  

/tmp/ipykernel_1574/2271207170.py:473: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = __import__('datetime').datetime.utcnow().isoformat()


False

## Cell 11 — Head-to-Head Comparison & Model Selection

In [24]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
print("=" * 72)
print("  STAGE 3 v2 — VAE-A vs VAE-B COMPARISON")
print("=" * 72)
print(f"\n  {'Metric':<30} {'VAE-A (167)':>14} {'VAE-B (140)':>14} {'Target':>14}")
print("  " + "-" * 70)

metrics = [
    ("val_recon (best)",     f"{ckpt_a['val_recon']:.6f}",
                             f"{ckpt_b['val_recon']:.6f}",
                             "0.0005–0.0007"),
    ("val_kl",               f"{ckpt_a['val_kl']:.3f}",
                             f"{ckpt_b['val_kl']:.3f}",
                             "8.0–14.0"),
    ("Min class distance",   f"{val_results_a['min_class_dist']:.4f}",
                             f"{val_results_b['min_class_dist']:.4f}",
                             "1.5–2.2"),
    ("Per-dim std (min)",    f"{min(val_results_a['dim_std']):.4f}",
                             f"{min(val_results_b['dim_std']):.4f}",
                             "0.60–0.95"),
    ("Collapsed dims",       f"{len(val_results_a['collapsed_dims'])}",
                             f"{len(val_results_b['collapsed_dims'])}",
                             "0"),
    ("Best epoch",           f"{ckpt_a['epoch']}",
                             f"{ckpt_b['epoch']}",
                             "—"),
    ("Parameters",           f"{sum(p.numel() for p in model_a.parameters()):,}",
                             f"{sum(p.numel() for p in model_b.parameters()):,}",
                             "—"),
    ("MALWARE recon loss",   f"{pcl_a.get('MALWARE',0):.6f}",
                             f"{pcl_b.get('MALWARE',0):.6f}",
                             "low"),
    ("EXPLOIT recon loss",   f"{pcl_a.get('EXPLOIT',0):.6f}",
                             f"{pcl_b.get('EXPLOIT',0):.6f}",
                             "low"),
]
for row in metrics:
    print(f"  {row[0]:<30} {row[1]:>14} {row[2]:>14} {row[3]:>14}")

print()
print("  Closest pair VAE-A:", val_results_a['min_class_pair'])
print("  Closest pair VAE-B:", val_results_b['min_class_pair'])

# ── Provisional winner selection ──────────────────────────────────────────────
# Primary criterion: min_class_distance (directly predicts VQC separability)
# Secondary: val_recon (lower = encoder learned cleaner features)
# Final selection confirmed in Stage 4 by VQC macro F1.

a_dist = val_results_a['min_class_dist']
b_dist = val_results_b['min_class_dist']
a_recon = ckpt_a['val_recon']
b_recon = ckpt_b['val_recon']

if a_dist > b_dist:
    provisional = 'VAE-A'
    reason = f"Better class separation ({a_dist:.4f} vs {b_dist:.4f})"
elif b_dist > a_dist:
    provisional = 'VAE-B'
    reason = f"Better class separation ({b_dist:.4f} vs {a_dist:.4f})"
else:
    # Tie on distance — prefer lower reconstruction loss
    provisional = 'VAE-A' if a_recon < b_recon else 'VAE-B'
    reason = f"Tie on class distance — lower val_recon wins"

print(f"\n  Provisional winner: {provisional}  ({reason})")
print(f"  Final selection:    Run Stage 4 VQC on BOTH — pick higher macro F1")

# ── Write stage3_selected_model.json ─────────────────────────────────────────
selection = {
    'provisional_winner':   provisional,
    'selection_criterion':  'VQC macro F1 on full 573K test set (Stage 4)',
    'vae_a': {
        'input_dim':       INPUT_DIM_A,
        'best_epoch':      ckpt_a['epoch'],
        'val_recon':       ckpt_a['val_recon'],
        'val_kl':          ckpt_a['val_kl'],
        'min_class_dist':  a_dist,
        'z_train_path':    os.path.join(STAGE3_DIR_A, 'vae_a_z_train.parquet'),
        'z_test_path':     os.path.join(STAGE3_DIR_A, 'vae_a_z_test.parquet'),
        'output_dir':      STAGE3_DIR_A,
    },
    'vae_b': {
        'input_dim':       INPUT_DIM_B,
        'best_epoch':      ckpt_b['epoch'],
        'val_recon':       ckpt_b['val_recon'],
        'val_kl':          ckpt_b['val_kl'],
        'min_class_dist':  b_dist,
        'z_train_path':    os.path.join(STAGE3_DIR_B, 'vae_b_z_train.parquet'),
        'z_test_path':     os.path.join(STAGE3_DIR_B, 'vae_b_z_test.parquet'),
        'output_dir':      STAGE3_DIR_B,
    },
    'class_weights_used': CLASS_WEIGHTS_DICT,
    'smote_version': 'v2',
}

sel_path = os.path.join(notebook_dir, 'stage3_selected_model.json')
with open(sel_path, 'w') as f:
    json.dump(selection, f, indent=2)
print(f"\n  Written: {sel_path}")

  STAGE 3 v2 — VAE-A vs VAE-B COMPARISON

  Metric                            VAE-A (167)    VAE-B (140)         Target
  ----------------------------------------------------------------------
  val_recon (best)                     0.000333       0.000350  0.0005–0.0007
  val_kl                                 16.000         16.000       8.0–14.0
  Min class distance                     2.4497         2.1685        1.5–2.2
  Per-dim std (min)                      0.6584         0.6117      0.60–0.95
  Collapsed dims                              0              0              0
  Best epoch                                 82            111              —
  Parameters                            507,319        479,644              —
  MALWARE recon loss                   0.001767       0.001897            low
  EXPLOIT recon loss                   0.002208       0.002351            low

  Closest pair VAE-A: ['EXPLOIT', 'MALWARE']
  Closest pair VAE-B: ['EXPLOIT', 'MALWARE']

  Provisional

## Cell 12 — Final Summary & Stage 4 Readiness

In [25]:
print("=" * 72)
print("  STAGE 3 v2 — COMPLETE")
print("=" * 72)

for tag, sd, cd, vr, hist, z_tr, z_te in [
    ('VAE-B', STAGE3_DIR_B, CONFIG_B, ckpt_b, history_b, z_b_train, z_b_test),
    ('VAE-A', STAGE3_DIR_A, CONFIG_A, ckpt_a, history_a, z_a_train, z_a_test),
]:
    dim_std = z_tr.std(dim=0)
    print(f"\n  {tag}  ({cd['input_dim']} features  -> 8D latent)")
    print(f"    Output folder:   {sd}")
    print(f"    Best epoch:      {vr['epoch']} / {cd['max_epochs']}")
    print(f"    val_recon:       {vr['val_recon']:.6f}  (target 0.0005-0.0007)")
    print(f"    val_kl:          {vr['val_kl']:.4f}      (target 8.0-14.0)")
    print(f"    z_train shape:   {list(z_tr.shape)}")
    print(f"    z_test  shape:   {list(z_te.shape)}")
    print(f"    Angle range:     [{z_tr.min():.4f}, {z_tr.max():.4f}]")
    print(f"    Per-dim std min: {dim_std.min():.4f}  (target 0.60-0.95)")
    print(f"    Class weights:   v2 applied in aux loss  (Malware=47.6x Normal)")

print("  Class weights used (v2):")
print("    Normal=0.2626  DoS=1.5220  Recon=3.1747  Exploit=5.0127  Malware=47.6208")
print()
print("  Output directories:")
print(f"    vae_b_output/  ({CONFIG_B[chr(39)+'input_dim'+chr(39)]} features WITHOUT near-zero)")
print(f"    vae_a_output/  ({CONFIG_A[chr(39)+'input_dim'+chr(39)]} features WITH near-zero)")
print()
print("  Stage 4: load z_train.parquet and z_test.parquet from each folder.")
print("  Use stage2_class_weights.json for weighted VQC loss.")
print("  Run Stage 4 on BOTH models independently.")
print("  Compare macro F1 on full 573807-sample test set.")
print("  Winning model feeds Stage 5 ensemble.")
print("=" * 72)
log_gpu('final')

  STAGE 3 v2 — COMPLETE

  VAE-B  (140 features  -> 8D latent)
    Output folder:   /MINOR_PROJECT/vae_b_output
    Best epoch:      111 / 120
    val_recon:       0.000350  (target 0.0005-0.0007)
    val_kl:          16.0000      (target 8.0-14.0)
    z_train shape:   [2381042, 8]
    z_test  shape:   [573807, 8]
    Angle range:     [0.0000, 3.1416]
    Per-dim std min: 0.6117  (target 0.60-0.95)
    Class weights:   v2 applied in aux loss  (Malware=47.6x Normal)

  VAE-A  (167 features  -> 8D latent)
    Output folder:   /MINOR_PROJECT/vae_a_output
    Best epoch:      82 / 120
    val_recon:       0.000333  (target 0.0005-0.0007)
    val_kl:          16.0000      (target 8.0-14.0)
    z_train shape:   [2381042, 8]
    z_test  shape:   [573807, 8]
    Angle range:     [0.0000, 3.1416]
    Per-dim std min: 0.6584  (target 0.60-0.95)
    Class weights:   v2 applied in aux loss  (Malware=47.6x Normal)
  Class weights used (v2):
    Normal=0.2626  DoS=1.5220  Recon=3.1747  Exploit=5.012

KeyError: "'input_dim'"

## Cell 13 — Package Outputs for Download

In [26]:
import shutil

for tag, d in [('vae_a_output_16', '/MINOR_PROJECT/vae_a_output_16'), ('vae_b_output_16', '/MINOR_PROJECT/vae_b_output_16')]:
    zip_path = os.path.join(notebook_dir, f'{tag}.zip')
    shutil.make_archive(os.path.join(notebook_dir, tag), 'zip', d)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"  {tag}.zip  ({size_mb:.1f} MB)  ->  {zip_path}")

print("\nDownload both zips to your local machine.")
print("Stage 4 only needs the parquet files and stage3_selected_model.json.")

  vae_a_output_16.zip  (105.4 MB)  ->  /MINOR_PROJECT/vae_a_output_16.zip
  vae_b_output_16.zip  (104.7 MB)  ->  /MINOR_PROJECT/vae_b_output_16.zip

Download both zips to your local machine.
Stage 4 only needs the parquet files and stage3_selected_model.json.
